# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [40]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
5161,47093,BYRNE DAIRY #40,7767 ST RT 5,CLINTON,13323,82,CONV. STORE,-75.37354,43.07919,b'\x01\x01\x00\x00\x00\xdd\xefP\x14\xe8\xd7R\x...,360650250011,36065025001
6476,68081,FARMINGVILLE MEAT & PRODUCE,1105 HORSEBLOCK RD,FARMINGVILLE,11738,0,GROCERY STORE,-73.02233,40.83096,b'\x01\x01\x00\x00\x00\x0f\xee\xce\xdamAR\xc0_...,361031585101,36103158510
14155,113460,P&P LIQUORS,530A MERRICK RD,ROCKVILLE CENTRE,11570,0,LIQUOR,-73.63407,40.65446,b'\x01\x01\x00\x00\x00\xfbWV\x9a\x94hR\xc0$EdX...,360594123021,36059412302
7570,76930,STOP&GO CONVENIENCE & SMOKE SH,485 HEMPSTEAD TPKE,WEST HEMPSTEAD,11552,0,CONV. STORE,-73.65739,40.70541,b'\x01\x01\x00\x00\x00\xeb\xad\x81\xad\x12jR\x...,360594060013,36059406001
16615,113947,CORNER BAKERY INC,8402 QUEENS BLVD,ELMHURST,11373,0,SPECIALTY (FOOD),-73.87927,40.73694,b'\x01\x01\x00\x00\x00\xa7\x96\xad\xf5ExR\xc0\...,360810473002,36081047300


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
6082,114697,26394.0,2025-01-02,2025-12-30,2025-01-02,2025-12-30,201.0,363,363.0,0.0,0.0,72.710744,72.710744,False,72.710744
191,100920,673649.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1845.615068,1845.615068,False,1845.615068
6618,115318,443339.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1214.628767,1214.628767,False,1214.628767
1832,106285,1118313.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,3063.872603,3063.872603,False,3063.872603
6843,115563,62405.0,2025-01-10,2025-12-31,2025-01-10,2025-12-31,341.0,356,356.0,0.0,0.0,175.294944,175.294944,False,175.294944


## Overture Maps Data

In [53]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['food_and_drink' 'services_and_business' 'sports_and_recreation'
 'education' 'cultural_and_historic' 'health_care'
 'arts_and_entertainment' 'shopping' 'travel_and_transportation'
 'lifestyle_services' 'community_and_government' 'geographic_entities'
 'lodging' None]
202


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
952,2a1417f5-c2e8-48ad-96b2-545ee0d940e7,b'\x01\x01\x00\x00\x00x\xf3\xaf@\xcf*R\xc0%\xa9\xd0i\xabuD@',-72.668900,40.919294,dental_clinic,orthodontics,"[health_care, outpatient_care_facility, dental_clinic, orthodontics]",0.990219,NaN,2025-06-25.0,health_care
2519,031f3ad0-d5cb-455f-a365-d526e08ee9c5,b'\x01\x01\x00\x00\x00\x07\xbf\xa2\x95\x1a-R\xc0\xe6\xa4 \r\xe1hD@',-72.704748,40.819368,fitness_studio,boot_camp,"[sports_and_recreation, sport_or_fitness_facility, fitness_studio, boot_camp]",0.990219,NaN,2025-06-25.0,sports_and_recreation
4295,eefd1c6e-c032-435c-a765-6ccee41f31df,b'\x01\x01\x00\x00\x00\xd6T\x16\x85]xR\xc0\x88\xf3p\x02\xd3\xfdD@',-73.880708,41.983002,casual_eatery,sandwich_shop,"[food_and_drink, casual_eatery, sandwich_shop]",0.989402,NaN,2025-06-25.0,food_and_drink
3892,7d620973-cf0d-46f1-a7d7-6033fa1819bf,b'\x01\x01\x00\x00\x00\x14\xb1\x88a\x87*R\xc0` \x08\x90\xa1wD@',-72.664513,40.934618,fashion_and_apparel_store,jewelry_store,"[shopping, fashion_and_apparel_store, jewelry_store]",0.919912,NaN,2025-06-25.0,shopping
456,1666e15e-0b69-4e55-a4e1-6bc7a1fc8bea,b'\x01\x01\x00\x00\x00\\*F\xc8\x861R\xc0\x7f2\x08\xa4\xabvD@',-72.773851,40.927113,hardware_home_and_garden_store,hardware_store,"[shopping, specialty_store, hardware_home_and_garden_store, hardware_store]",0.999871,Tractor Supply Co.,2025-06-25.0,shopping
374,269e4c44-8641-4c09-87f7-4a63a067a830,b'\x01\x01\x00\x00\x00^+\xa1\xbb$3R\xc0\xeb\xaa@-\x06mD@',-72.799117,40.851751,physical_medicine_and_rehabilitation,physical_therapy,"[health_care, outpatient_care_facility, physical_medicine_and_rehabilitation, physical_therapy]",0.919912,NaN,2025-06-25.0,health_care
1675,40b76ef4-42c5-444f-bac7-529c41212d58,b'\x01\x01\x00\x00\x00L\xfa{)<&R\xc0\rnk\x0b\xcfkD@',-72.597422,40.842256,taxi_or_ride_share_service,limo_service,"[travel_and_transportation, ground_transport_facility_or_service, taxi_or_ride_share_service, limo_service]",0.850000,NaN,2025-06-25.0,travel_and_transportation
3047,87e7e159-ffb3-4f62-91af-75515da404f7,b'\x01\x01\x00\x00\x00\x19\xab\x8b\xe3\x99=R\xc0\x05\x80S\xed\x8dyD@',-72.962518,40.949644,arts_crafts_and_hobby_store,fabric_store,"[shopping, specialty_store, arts_crafts_and_hobby_store, fabric_store]",0.919912,NaN,2025-06-25.0,shopping
4128,1fe1d674-04b2-4330-bf2b-cd49b96a941a,b'\x01\x01\x00\x00\x00\xaf\xa8\xb1R;#R\xc0/\xe2u\xdct~D@',-72.550496,40.987941,brewery,brewery,"[food_and_drink, alcoholic_beverage_venue, brewery]",0.947707,NaN,2025-06-25.0,food_and_drink
800,07a2d27f-764d-42fc-9edb-2d1f9e0befce,"b'\x01\x01\x00\x00\x00)\xe5\x18=5,R\xc0\xed%\x11\n\xdfvD@'",-72.690749,40.928682,arts_crafts_and_hobby_store,arts_crafts_and_hobby_store,"[shopping, specialty_store, arts_crafts_and_hobby_store]",0.919912,Hobby Lobby,2025-06-25.0,shopping


In [7]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [8]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
53,112373,SOF DELI CORP,6803 3RD AVE,BROOKLYN,11220,-74.025840,40.636260,b'\x01\x01\x00\x00\x00o\xbb\xd0\\\xa7\x81R\xc0...,360470068004,36047006800,3,6,2,4,2,8,7,23,5,21,4,8
16,14321,METRO LOBBY NEWSSTAND,1585 BROADWAY,NEW YORK,10036,-73.985020,40.759610,b'\x01\x01\x00\x00\x00\xfbyS\x91\n\x7fR\xc0w\x...,360610125004,36061012500,2,6,49,123,42,143,128,444,75,198,61,130
66,72992,PORT JEFF GRAPES & GRAINS,514 JEFFERSON PLAZA,PORT JEFF STA,11776,-73.052214,40.930503,b'\x01\x01\x00\x00\x00-\x99cyWCR\xc0:\xea\xe8\...,361031582091,36103158209,3,6,0,0,0,1,3,9,1,2,1,1
63,89815,AL'S DELI & GROCERY I CORP,160-26 BAISLEY BLVD,JAMAICA,11434,-73.781430,40.676990,b'\x01\x01\x00\x00\x00@\x87\xf9\xf2\x02rR\xc0\...,360810294001,36081029400,4,13,0,3,1,1,0,9,1,7,0,1
74,111311,ADK FOOD & FUEL,1 COUNTRY MILE,FRANKFORT,13340,-75.088290,43.038500,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6\xc5...,360430102022,36043010202,0,2,1,1,0,0,0,0,0,0,0,1


## LEHD Data

In [9]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [10]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
83,108868,WEST44TH STREET NEWSSTAND LLC,300 WEST 44TH STREET,NEW YORK,10036,-73.98944,40.75879,b'\x01\x01\x00\x00\x00\xacV&\xfcR\x7fR\xc0\x13...,360610121012,36061012101,1044.0,151.0,120.0,773.0
51,88721,A TO Z GROCERY & DELI INC,1450 CROSBY AVE,BRONX,10461,-73.83000,40.84207,b'\x01\x01\x00\x00\x00\x85\xebQ\xb8\x1euR\xc0\...,360050264001,36005026400,748.0,104.0,181.0,463.0


In [11]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [12]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
14,108540,DON VINO LIQUORS LLC,95-08B 35TH AVENUE,JACKSON HEIGHTS,11372,-73.87235,40.75320,b'\x01\x01\x00\x00\x00\x99*\x18\x95\xd4wR\xc0q...,360810273023,36081027302,344.0,280.0,42.0,32.0,169.0,143.0
45,111759,TT2 TRADING INC,376 S BAYVIEW AVE,FREEPORT,11520,-73.59283,40.64398,b'\x01\x01\x00\x00\x00\x9d\x85=\xed\xf0eR\xc0\...,360594145012,36059414501,342.0,43.0,22.0,60.0,93.0,189.0


## AADT Data

In [8]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [12]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

9759 16801


,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
6434,48515,CLIFF'S LOCAL MARKET #514,11468 STATE RT 12,ALDER CREEK,13301,-75.224397,43.415909,360650242001,36065024200,54296.0,295.0,EGYPT RD,09,1:1||||,872.374730
7488,114280,FULL CIRCLE BAR AND GRILL,616 LERAY ST,WATERTOWN,13601,-75.912530,43.983210,360450614004,36045061400,NaN,NaN,NaN,NaN,NaN,NaN
16242,111938,LIGHTHOUSE PHARMACY,747 MONTAUK HWY,PATCHOGUE,11772,-72.987360,40.767720,361031591103,36103159110,NaN,NaN,NaN,NaN,NaN,NaN
10137,64336,SPEEDWAY #07804,550 ATLANTIC AVE,OCEANSIDE,11572,-73.656080,40.641120,360594130013,36059413001,NaN,NaN,NaN,NaN,NaN,NaN
4613,113734,7-ELEVEN #18249F,266 HOOKER AVENUE,POUGHKEEPSIE,12603,-73.909950,41.686260,360272209014,36027220901,55064.0,1940.0,S GRAND AVE,19,1:1||||,51.836833
1215,110976,RS FOOD MART INC,804 LAKE ST,ELMIRA,14901,-76.804140,42.100260,360150001003,36015000100,54832.0,9214.0,E Water St,14,1:1||||,1346.713675
4690,80462,STOP & SHOP #0503,5716 BROADWAY,BRONX,10463,-73.902010,40.881870,360050283002,36005028300,54438.0,1816.0,W 238TH ST,17,1:1||||,354.356161
11348,77789,MIRABITO #83-GLEN AUBREY,2965 NYS RT 26,GLEN AUBREY,13777,-76.035590,42.214440,360070102001,36007010200,NaN,NaN,NaN,NaN,NaN,NaN
2042,114942,C-TOWN,2807 86TH ST,BROOKLYN,11223,-73.978740,40.592810,360470308001,36047030800,55646.0,5927.0,BENSON AVE,16,1:1||||,506.110786
11475,79515,TOURE ENTERPRISE,905 COLUMBUS AVE,NEW YORK,10025,-73.963700,40.798120,360610189005,36061018900,NaN,NaN,NaN,NaN,NaN,NaN


## EPA Data

In [15]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [16]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
56,110788,SUNRISE 99 CENT STORE INC,4008 QUEENS BLVD,SUNNYSIDE,11104,0,VARIETY,-73.92392,40.74361,b'\x01\x01\x00\x00\x00\xb3\x07Z\x81!{R\xc0A}\x...,360810181013,36081018101,162297,36,81,18101,3,73.247865,137.816070,33.448447,2.857937,2.857937,24.345389,1.376044,1.270194,0.529248,0.211699,0.0,39.499071,12.500000,36,081,018101,3
20,113105,NAZ DELI CORP,234 E 106TH ST,NEW YORK,10029,0,SUPERMARKETS,-73.94326,40.79098,"b""\x01\x01\x00\x00\x00\xff\xe70_^|R\xc0!\xe5'\...",360610170001,36061017000,153070,36,61,17000,1,56.809486,105.767254,30.560116,2.309329,6.235187,0.153955,1.616530,0.000000,18.628585,1.616530,0.0,17.201267,10.666667,36,061,017000,1
64,113075,RAINBOW DELI GROCERY CORP,38-15 GREENPOINT AVE,LONG ISLAND CITY,11101,0,DELI-GROCERY,-73.92779,40.73838,b'\x01\x01\x00\x00\x00\x93\xe3N\xe9`{R\xc0\x9f...,360810189003,36081018900,149243,36,81,18900,3,27.428359,70.809118,18.068534,0.081390,0.162780,1.220847,1.383626,0.244169,13.673485,1.302237,0.0,42.892341,15.833333,36,081,018900,3
28,115205,ABE GRILL & MINI MART INC,1160 YONKERS AVE,YONKERS,10704,0,CONV. STORE,-73.85342,40.91390,b'\x01\x01\x00\x00\x00&p\xebn\x9evR\xc0>\xe8\x...,361190015051,36119001505,153582,36,119,1505,1,19.960106,34.390499,2.125535,0.066423,0.232480,0.016606,0.381932,0.963133,0.000000,0.464961,0.0,30.773597,12.333333,36,119,001505,1
76,78437,CITY MARKET & DELI INC,1245 EAST DELAVAN AVE,BUFFALO,14215,0,CONV. STORE,-78.81093,42.92269,b'\x01\x01\x00\x00\x00\x19V\xf1F\xe6\xb3S\xc0R...,360290041003,36029004100,159797,36,29,4100,3,6.600410,15.409228,1.414374,0.769221,0.397017,0.000000,0.223322,0.024814,0.000000,0.000000,0.0,25.981046,14.166667,36,029,004100,3


## ACS Data

In [17]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [18]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
46,39785,DANIELS DELI,2554 SOUTH AVENUE STORE 2,WAPPINGERS FALLS,12590,0,DELI-GROCERY,-73.91967,41.59167,b'\x01\x01\x00\x00\x000G\x8f\xdf\xdbzR\xc0\x1d...,360273000004,36027300000,76723.0,519.0,1174.0,0.219761,0.476150,0.304089,0.026658,0.188272,0.160494,0.532609,0.103499,0.527938,0.472062,0.653179,0.052023,0.294798,0.052023
48,77604,GEORGIA DELI,272 GEORGIA ST,BUFFALO,14201,0,GROCERY STORE,-78.87996,42.89121,b'\x01\x01\x00\x00\x00qr\xbfCQ\xb8S\xc0 \xefU+...,360290071022,36029007102,NaN,480.0,675.0,0.198519,0.648889,0.152593,0.293051,0.250804,0.081994,0.459259,0.000000,0.233333,0.766667,0.214583,0.168750,0.616667,0.489583
75,113075,RAINBOW DELI GROCERY CORP,38-15 GREENPOINT AVE,LONG ISLAND CITY,11101,0,DELI-GROCERY,-73.92779,40.73838,b'\x01\x01\x00\x00\x00\x93\xe3N\xe9`{R\xc0\x9f...,360810189003,36081018900,77803.0,338.0,549.0,0.183971,0.577413,0.238616,0.089530,0.433515,0.100182,0.808743,0.090090,0.292899,0.707101,0.452663,0.124260,0.423077,0.556213
63,100319,MIA TRAVEL MULTISERVICE,5 E CLARK PLACE,BRONX,10452,0,TRAVEL AGENT,-73.91877,40.83828,b'\x01\x01\x00\x00\x00\xf0\xa2\xaf \xcdzR\xc0\...,360050221013,36005022101,65609.0,625.0,1319.0,0.388931,0.595906,0.015163,0.319920,0.143275,0.000000,0.653651,0.154867,0.000000,1.000000,0.872000,0.060800,0.067200,0.721600
57,108836,SHUNFA DISCOUNT INC,5309 CHURCH AVE,BROOKLYN,11203,0,GROCERY STORE,-73.92711,40.65235,b'\x01\x01\x00\x00\x00\xdar.\xc5U{R\xc0\xd6\xc...,360470866003,36047086600,NaN,512.0,1045.0,0.113876,0.736842,0.149282,0.233102,0.287368,0.116842,0.688910,0.145975,0.156250,0.843750,0.462891,0.025391,0.511719,0.767578


## Master Data Sample

In [16]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(16801, 69)
Index(['agtno', 'busnm', 'busaddr', 'buscity', 'buszip', 'chain', 'bustype',
       'longitude', 'latitude', 'geometry', 'bg_geoid', 'tract_geoid',
       'aadt_count', 'aadt_road_name', 'aadt_functional_class', 'aadt_lanes',
       'dist_to_read_m', 'median_household_income', 'households_total',
       'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'age_65_plus_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'renter_occupied_proportion', 'family_households_proportion',
       'nonfamily_multiperson_households_proportion',
       'living_alone_households_proportion',
       'no_vehicle_households_proportion', 'D1A', 'D1B', 'D1C', 'D1C8_RET',
       'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT', 'D1C8_ED', 'D1C8_HLTH',
       'D1C8_PUB', 'D3A', 'NatWalkInd', 'gas_station_within_0.

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
16321,116553,PARISH FOOD MART INC,2886 E MAIN ST,PARISH,13131,0,CONV. STORE,-76.12004,43.40535,b'\x01\x01\x00\x00\x00\x8f\x8d@\xbc\xae\x07S\x...,360750205022,36075020502,NaN,NaN,NaN,NaN,NaN,72389.0,690.0,1434.0,0.155509,0.602510,0.241980,0.119714,0.095275,0.043377,0.590909,0.082854,0.804348,0.195652,0.633333,0.115942,0.250725,0.092754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,788.0,135.0,186.0,467.0,510.0,34.0,45.0,82.0,138.0,290.0
15702,115742,7-ELEVEN #35495D,42-02 QUEEN BLVD,SUNNYSIDE,11104,30,CONV. STORE,-73.92204,40.74315,b'\x01\x01\x00\x00\x00\xa5f\x0f\xb4\x02{R\xc0\...,360810181022,36081018102,NaN,NaN,NaN,NaN,NaN,89702.0,496.0,829.0,0.143546,0.782871,0.073583,0.042553,0.371728,0.244764,0.656036,0.105903,0.010081,0.989919,0.526210,0.120968,0.352823,0.745968,70.372511,114.276555,20.376518,8.822822,1.575504,1.890605,2.205706,3.256042,2.415773,0.210067,0.0,21.674202,14.666667,4.0,15.0,3.0,5.0,4.0,17.0,6.0,38.0,10.0,19.0,9.0,18.0,757.0,98.0,135.0,524.0,143.0,55.0,15.0,29.0,59.0,55.0
2580,89485,TSM GROCERY,2285 1ST AVE,NEW YORK,10035,0,GROCERY STORE,-73.93469,40.79701,b'\x01\x01\x00\x00\x00}y\x01\xf6\xd1{R\xc0\xe0...,360610188001,36061018800,57017.0,WILLIS AVE BRG,14,4:0||||,601.811277,27083.0,475.0,706.0,0.158640,0.480170,0.361190,0.300122,0.101983,0.123229,0.624646,0.088435,0.073684,0.926316,0.454737,0.000000,0.545263,0.981053,57.427891,101.081974,12.551937,0.111079,0.111079,0.000000,0.333237,1.444028,10.219276,0.333237,0.0,33.512819,8.333333,3.0,9.0,11.0,23.0,4.0,10.0,26.0,79.0,8.0,26.0,2.0,10.0,422.0,78.0,111.0,233.0,155.0,10.0,27.0,15.0,38.0,102.0
4401,36799,STEWART'S SHOPS #425,201 N COMRIE AVENUE,JOHNSTOWN,12095,23,CONV. STORE,-74.36083,43.01386,b'\x01\x01\x00\x00\x00\x9bZ\xb6\xd6\x17\x97R\x...,360359712001,36035971200,NaN,NaN,NaN,NaN,NaN,42200.0,554.0,964.0,0.258299,0.485477,0.256224,0.147929,0.100719,0.037170,0.476141,0.000000,0.925993,0.074007,0.339350,0.045126,0.615523,0.090253,0.588971,1.265471,1.959774,0.474737,0.244786,0.583036,0.142421,0.351602,0.004451,0.158740,0.0,12.444875,10.166667,3.0,4.0,0.0,0.0,2.0,2.0,1.0,3.0,0.0,3.0,0.0,3.0,542.0,94.0,159.0,289.0,1220.0,328.0,194.0,283.0,444.0,493.0
14659,114285,JULITIN DELI GROCERY CORP,1290 SUTTER AVE,BROOKLYN,11208,0,SUPERMARKETS,-73.87050,40.67298,b'\x01\x01\x00\x00\x00\xc1\xca\xa1E\xb6wR\xc0Y...,360471208011,36047120801,NaN,NaN,NaN,NaN,NaN,NaN,223.0,739.0,0.144790,0.687415,0.167794,0.000000,0.134494,0.058544,0.825440,0.000000,0.206278,0.793722,0.560538,0.152466,0.286996,0.860987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,7.0,1.0,5.0,4.0,7.0,4.0,12.0,1.0,11.0,0.0,3.0,340.0,64.0,111.0,165.0,10.0,6.0,0.0,5.0,5.0,0.0


## Missingness and Variance Check

In [31]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [32]:
numeric_features = ['aadt_count', 'dist_to_read_m',
                    'median_household_income', 'households_total', 'total_population',
                    'age_18_29_proportion', 'age_30_64_proportion', 'age_65_plus_proportion',
                    'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
                    'labor_force_participation_rate', 'unemployment_rate',
                    'owner_occupied_proportion', 'renter_occupied_proportion',
                    'family_households_proportion', 'nonfamily_multiperson_households_proportion',
                    'living_alone_households_proportion', 'no_vehicle_households_proportion',
                    'D1A', 'D1B', 'D1C', 'D1C8_RET', 'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT',
                    'D1C8_ED', 'D1C8_HLTH', 'D1C8_PUB', 'D3A', 'NatWalkInd',
                    'gas_station_within_0.5_mi', 'gas_station_within_1_mi', 'arts_and_entertainment_within_0.5_mi',
                    'arts_and_entertainment_within_1_mi', 'shopping_within_0.5_mi',
                    'shopping_within_1_mi', 'social_or_community_service_within_0.5_mi',
                    'social_or_community_service_within_1_mi', 'restaurant_within_0.5_mi',
                    'restaurant_within_1_mi', 'bar_within_0.5_mi', 'bar_within_1_mi',
                    'rac_c000', 'rac_ce01', 'rac_ce02', 'rac_ce03', 'wac_c000', 'wac_cns07',
                    'wac_cns18', 'wac_ce01', 'wac_ce02', 'wac_ce03']

categorical_features = ['chain', 'bustype', 'aadt_functional_class', 'aadt_lanes']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
aadt_count,58.085828,0.000000,3.180915,5466.000000,5865.659472,8804.543594,5334.000000,1.501032,563
dist_to_read_m,56.966847,0.000000,0.082988,627.332754,778.900103,445.966603,744.678223,0.572559,6606
D1A,18.314386,0.261889,0.320606,0.000000,17.706218,28.001488,22.008052,1.581449,6444
D1B,18.314386,0.202369,0.247741,0.000000,42.559408,64.675765,56.678018,1.519658,6450
D1C,18.314386,0.136897,0.167590,0.000000,30.103823,140.466582,11.276514,4.666071,6455
D1C8_RET,18.314386,4.821142,5.902069,0.000000,2.599771,10.403593,1.586033,4.001734,5926
D1C8_PUB,18.314386,66.728171,81.689012,0.000000,1.397457,20.067523,0.000000,14.360026,1000
D1C8_HLTH,18.314386,12.237367,14.981055,0.000000,4.933529,26.571133,1.387777,5.385827,5274
D1C8_ED,18.314386,40.955895,50.138444,0.000000,1.675417,12.355113,0.294076,7.374353,2937
D1C8_ENT,18.314386,9.713708,11.891577,0.000000,3.077399,14.310108,1.099186,4.650066,5432


In [ ]:
categorical_summary['aadt_functional_class']

{'pct_missing': np.float64(57.96083566454377),
 'distribution':                              pct
 aadt_functional_class           
 NaN                    57.960836
 19                     22.623653
 17                      6.332956
 16                      5.767514
 14                      4.184275
 09                      1.178501
 12                      0.755907
 18                      0.404738
 11                      0.255937
 08                      0.190465
 30                      0.136897
 07                      0.119041
 1                       0.035712
 9                       0.029760
 04                      0.017856
 8                       0.005952}

## Correlation

In [38]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951
